## HTML Executive Report

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

outputs_dir = Path().resolve().parent / 'outputs'

CAUSAL_CHAIN_SVG = """
<svg width="700" height="100" xmlns="http://www.w3.org/2000/svg" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="arrow" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#636e72"/>
    </marker>
  </defs>

  <!-- Boxes -->
  <rect x="0"   y="25" width="110" height="50" rx="6" fill="#dfe6e9"/>
  <rect x="150" y="25" width="110" height="50" rx="6" fill="#dfe6e9"/>
  <rect x="300" y="25" width="110" height="50" rx="6" fill="#dfe6e9"/>
  <rect x="450" y="25" width="110" height="50" rx="6" fill="#E8533B" opacity="0.85"/>
  <rect x="600" y="0"  width="100" height="100" rx="6" fill="#00b894" opacity="0.85"/>

  <!-- Labels -->
  <text x="55"  y="46" text-anchor="middle" font-size="10" fill="#2d3436">Unmanaged</text>
  <text x="55"  y="60" text-anchor="middle" font-size="10" fill="#2d3436">condition</text>
  <text x="205" y="46" text-anchor="middle" font-size="10" fill="#2d3436">No allied</text>
  <text x="205" y="60" text-anchor="middle" font-size="10" fill="#2d3436">health use</text>
  <text x="355" y="46" text-anchor="middle" font-size="10" fill="#2d3436">Condition</text>
  <text x="355" y="60" text-anchor="middle" font-size="10" fill="#2d3436">escalates</text>
  <text x="505" y="42" text-anchor="middle" font-size="10" fill="white" font-weight="bold">ED visit /</text>
  <text x="505" y="56" text-anchor="middle" font-size="10" fill="white" font-weight="bold">admission</text>
  <text x="505" y="70" text-anchor="middle" font-size="9"  fill="white">Avoidable cost</text>
  <text x="650" y="35" text-anchor="middle" font-size="10" fill="white" font-weight="bold">Nudge</text>
  <text x="650" y="50" text-anchor="middle" font-size="10" fill="white" font-weight="bold">intervention</text>
  <text x="650" y="65" text-anchor="middle" font-size="9"  fill="white">Use unused</text>
  <text x="650" y="78" text-anchor="middle" font-size="9"  fill="white">benefits</text>
  <text x="650" y="92" text-anchor="middle" font-size="9"  fill="white">→ cost avoided</text>

  <!-- Arrows -->
  <line x1="112" y1="50" x2="146" y2="50" stroke="#636e72" stroke-width="1.5" marker-end="url(#arrow)"/>
  <line x1="262" y1="50" x2="296" y2="50" stroke="#636e72" stroke-width="1.5" marker-end="url(#arrow)"/>
  <line x1="412" y1="50" x2="446" y2="50" stroke="#636e72" stroke-width="1.5" marker-end="url(#arrow)"/>
  <!-- Intervention arrow curves back from nudge to break the chain -->
  <path d="M600,50 Q575,15 560,25" stroke="#00b894" stroke-width="2" fill="none" marker-end="url(#arrow)" stroke-dasharray="4,3"/>
</svg>
"""

# --- Draw causal chain PNG directly with matplotlib (no system DLL deps) ---
fig, ax = plt.subplots(figsize=(9, 1.5))
ax.set_xlim(0, 700); ax.set_ylim(0, 100); ax.axis('off')
fig.patch.set_facecolor('white')

boxes = [
    (0,   25, 110, 50, '#dfe6e9', '#2d3436', ['Unmanaged', 'condition']),
    (150, 25, 110, 50, '#dfe6e9', '#2d3436', ['No allied', 'health use']),
    (300, 25, 110, 50, '#dfe6e9', '#2d3436', ['Condition', 'escalates']),
    (450, 25, 110, 50, '#E8533B', 'white',   ['ED visit /', 'admission', 'Avoidable cost']),
    (600, 0,  100, 100,'#00b894', 'white',   ['Nudge', 'intervention', 'Use unused', 'benefits', '→ cost avoided']),
]
for (x, y, w, h, fc, tc, lines) in boxes:
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=2',
                                          fc=fc, ec='none', alpha=0.9,
                                          transform=ax.transData))
    cy = y + h / 2 + (len(lines) - 1) * 5
    for i, line in enumerate(lines):
        ax.text(x + w/2, cy - i*10, line, ha='center', va='center',
                fontsize=7, color=tc,
                fontweight='bold' if i < 2 and fc != '#dfe6e9' else 'normal')

# Chain arrows
for x1, x2 in [(112, 148), (262, 298), (412, 448)]:
    ax.annotate('', xy=(x2, 50), xytext=(x1, 50),
                arrowprops=dict(arrowstyle='->', color='#636e72', lw=1.5))

# Nudge intervention arc
ax.annotate('', xy=(558, 27), xytext=(600, 50),
            arrowprops=dict(arrowstyle='->', color='#00b894', lw=2,
                            connectionstyle='arc3,rad=-0.5',
                            linestyle='dashed'))

plt.tight_layout(pad=0.1)
plt.savefig(str(outputs_dir / 'causal_chain.png'), dpi=192, bbox_inches='tight',
            facecolor='white')
plt.close()
print(f"Saved: {outputs_dir / 'causal_chain.png'}")

# SVG for HTML report — unchanged
with open(outputs_dir / 'causal_chain.svg', 'w', encoding='utf-8') as f:
    f.write(CAUSAL_CHAIN_SVG)
print(f"Saved: {outputs_dir / 'causal_chain.svg'}")

In [ ]:
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.io as pio
from pathlib import Path
from jinja2 import Environment

data_dir    = Path().resolve().parent / 'data'
outputs_dir = Path().resolve().parent / 'outputs'

# Load data
scored   = pd.read_csv(outputs_dir / 'scored_members.csv')
summary  = json.load(open(outputs_dir / 'scoring_summary.json'))
metrics  = json.load(open(outputs_dir / 'eval_metrics.json'))
feat_imp = pd.read_csv(outputs_dir / 'feature_importance.csv').head(10)
scored['condition_cluster'] = scored['condition_cluster'].astype(str)
print(f"Loaded {len(scored):,} members for report generation")

COLOURS = {'High': '#E8533B', 'Medium': '#F2A65A', 'Low': '#4A6FA5'}

# Chart 1: Risk tier donut
tier_counts = scored['risk_tier'].value_counts().reindex(['High','Medium','Low'])
fig_tier = px.pie(
    values=tier_counts.values, names=tier_counts.index, hole=0.55,
    color=tier_counts.index, color_discrete_map=COLOURS,
    title='Population Risk Tier Distribution'
)
fig_tier.update_traces(textposition='outside', textinfo='percent+label')

# Chart 2: Nudge rate by condition cluster
nudge_by_cluster = (
    scored.groupby('condition_cluster')['nudge_signal']
    .agg(['sum','count'])
    .assign(nudge_rate=lambda x: x['sum']/x['count'])
    .sort_values('nudge_rate', ascending=False)
    .reset_index()
)
fig_cluster = px.bar(
    nudge_by_cluster, x='condition_cluster', y='nudge_rate',
    color='nudge_rate', color_continuous_scale=['#4A6FA5','#E8533B'],
    title='Nudge Rate by Condition Cluster',
    labels={'nudge_rate': 'Nudge Rate', 'condition_cluster': 'Condition Cluster'},
    text=nudge_by_cluster['nudge_rate'].map('{:.1%}'.format)
)
fig_cluster.update_traces(textposition='outside')
fig_cluster.update_layout(coloraxis_showscale=False)

# Chart 3: Risk score distribution
fig_dist = px.histogram(
    scored, x='risk_score', nbins=40, color='risk_tier',
    color_discrete_map=COLOURS, title='Risk Score Distribution',
    barmode='overlay', opacity=0.7
)

# Chart 4: Recommended modality breakdown (nudge cohort only)
nudge_df = scored[scored['nudge_signal'] == 1]
mod_counts = nudge_df['recommended_modality'].value_counts().reset_index()
mod_counts.columns = ['modality', 'count']
fig_modality = px.pie(
    mod_counts, values='count', names='modality', hole=0.5,
    title='Recommended Nudge Modality (Nudge Cohort)',
    color_discrete_sequence=['#4A6FA5','#F2A65A','#E8533B','#00b894']
)
fig_modality.update_traces(textposition='outside', textinfo='percent+label')

# Chart 5: Allied health utilisation by plan type
plan_util = (
    scored.groupby('plan_type')['allied_health_utilisation_rate']
    .mean().reset_index()
    .sort_values('allied_health_utilisation_rate', ascending=False)
)
fig_util = px.bar(
    plan_util, x='plan_type', y='allied_health_utilisation_rate',
    title='Avg Allied Health Utilisation Rate by Plan Type',
    labels={'allied_health_utilisation_rate': 'Avg Utilisation Rate', 'plan_type': 'Plan Type'},
    color='plan_type',
    color_discrete_map={'Bronze':'#E8533B','Silver':'#F2A65A','Gold':'#4A6FA5'},
    text=plan_util['allied_health_utilisation_rate'].map('{:.1%}'.format)
)
fig_util.update_traces(textposition='outside')
fig_util.update_layout(showlegend=False)

# Chart 6: SHAP feature importance
fig_shap = px.bar(
    feat_imp.sort_values('mean_abs_shap'),
    x='mean_abs_shap', y='feature', orientation='h',
    title='Top 10 Risk Predictors (Mean |SHAP| Value)',
    labels={'mean_abs_shap': 'Mean |SHAP|', 'feature': 'Feature'},
    color='mean_abs_shap', color_continuous_scale=['#4A6FA5','#E8533B']
)
fig_shap.update_layout(coloraxis_showscale=False)

# Business case numbers
nudge_count     = summary['nudge_signal']
true_positives  = round(nudge_count * metrics['precision_top20pct'])
conversion_rate = 0.20
ed_cost         = 800
cost_avoidance  = round(true_positives * conversion_rate * ed_cost)

charts = {
    'tier':     pio.to_html(fig_tier,     full_html=False, include_plotlyjs='cdn'),
    'cluster':  pio.to_html(fig_cluster,  full_html=False, include_plotlyjs=False),
    'dist':     pio.to_html(fig_dist,     full_html=False, include_plotlyjs=False),
    'modality': pio.to_html(fig_modality, full_html=False, include_plotlyjs=False),
    'util':     pio.to_html(fig_util,     full_html=False, include_plotlyjs=False),
    'shap':     pio.to_html(fig_shap,     full_html=False, include_plotlyjs=False),
}
print("Charts generated")
print(f"Business case: {nudge_count:,} nudges → {true_positives:,} est. true positives → AUD ${cost_avoidance:,} est. cost avoidance")

In [ ]:
# Load causal chain SVG inline
with open(outputs_dir / 'causal_chain.svg') as f:
    causal_chain_svg = f.read()

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Allied Health Nudge — Executive Report</title>
<style>
  body       { font-family: 'Segoe UI', Arial, sans-serif; margin: 0; background: #f5f6fa; color: #2d3436; }
  .header    { background: #2d3436; color: white; padding: 40px 60px 30px; }
  .header h1 { margin: 0 0 8px; font-size: 28px; font-weight: 600; }
  .header p  { margin: 0; opacity: 0.7; font-size: 14px; }
  .kpis      { display: flex; gap: 20px; padding: 30px 60px 10px; flex-wrap: wrap; }
  .kpi       { background: white; border-radius: 10px; padding: 24px 32px; flex: 1; min-width: 160px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); border-left: 4px solid #4A6FA5; }
  .kpi.alert { border-left-color: #E8533B; }
  .kpi.warn  { border-left-color: #F2A65A; }
  .kpi.good  { border-left-color: #00b894; }
  .kpi h2    { margin: 0 0 6px; font-size: 32px; font-weight: 700; }
  .kpi p     { margin: 0; font-size: 13px; color: #636e72; }
  .section   { padding: 10px 60px 30px; }
  .section h2{ font-size: 18px; font-weight: 600; margin: 30px 0 12px;
               border-bottom: 2px solid #e0e0e0; padding-bottom: 6px; }
  .causal    { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); margin-bottom: 24px; overflow-x: auto; }
  .causal h3 { margin: 0 0 16px; font-size: 15px; color: #636e72; font-weight: 600; }
  .charts    { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
  .chart     { background: white; border-radius: 10px; padding: 16px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); }
  .chart.wide{ grid-column: 1 / -1; }
  .bizcase   { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); border-left: 4px solid #00b894; }
  .bizcase h3{ margin: 0 0 12px; font-size: 15px; font-weight: 600; color: #00b894; }
  .biz-row   { display: flex; gap: 32px; margin: 16px 0; flex-wrap: wrap; }
  .biz-num   { text-align: center; }
  .biz-num h3{ font-size: 28px; font-weight: 700; margin: 0; color: #2d3436; }
  .biz-num p { font-size: 12px; color: #636e72; margin: 4px 0 0; }
  .biz-note  { font-size: 12px; color: #636e72; margin-top: 12px; line-height: 1.6; }
  .model     { background: white; border-radius: 10px; padding: 24px 32px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.07); }
  .metrics   { display: flex; gap: 32px; margin: 16px 0; flex-wrap: wrap; }
  .metric    { text-align: center; }
  .metric h3 { font-size: 28px; font-weight: 700; margin: 0; color: #4A6FA5; }
  .metric p  { font-size: 12px; color: #636e72; margin: 4px 0 0; }
  footer     { text-align: center; padding: 30px; color: #b2bec3; font-size: 12px; }
</style>
</head>
<body>

<div class="header">
  <h1>Allied Health Nudge — Executive Summary</h1>
  <p>Member risk scoring pipeline &middot; {{ total_scored }} members scored &middot; Synthetic dataset proof of concept</p>
</div>

<div class="kpis">
  <div class="kpi alert"><h2>{{ high_risk }}</h2><p>High Risk Members</p></div>
  <div class="kpi warn"><h2>{{ medium_risk }}</h2><p>Medium Risk Members</p></div>
  <div class="kpi"><h2>{{ nudge_count }}</h2><p>Nudge Signals Fired</p></div>
  <div class="kpi"><h2>{{ nudge_rate }}</h2><p>Nudge Rate</p></div>
  <div class="kpi good"><h2>{{ cost_avoidance }}</h2><p>Est. Cost Avoidance (AUD)<br><small>@ 20% conversion, $800/ED visit</small></p></div>
</div>

<div class="section">
  <h2>Why This Matters — The Causal Chain</h2>
  <div class="causal">
    <h3>Unmanaged conditions escalate when members don't use their allied health benefits</h3>
    {{ causal_chain_svg }}
    <p style="font-size:12px;color:#636e72;margin-top:12px;">
      The nudge intervention (green) breaks the chain before escalation reaches the ED.
      Members are only nudged when they have a Medium or High risk score <strong>and</strong>
      unused benefits remaining — every outreach has a concrete, actionable offer.
    </p>
  </div>

  <h2>Risk Distribution</h2>
  <div class="charts">
    <div class="chart">{{ charts.tier }}</div>
    <div class="chart">{{ charts.dist }}</div>
  </div>

  <h2>Nudge Cohort Profile</h2>
  <div class="charts">
    <div class="chart">{{ charts.cluster }}</div>
    <div class="chart">{{ charts.modality }}</div>
  </div>

  <h2>Plan Design Insight</h2>
  <div class="charts">
    <div class="chart wide">{{ charts.util }}</div>
  </div>

  <h2>Business Case</h2>
  <div class="bizcase">
    <h3>Estimated intervention value</h3>
    <div class="biz-row">
      <div class="biz-num"><h3>{{ nudge_count }}</h3><p>Members nudged</p></div>
      <div class="biz-num"><h3>{{ true_positives }}</h3><p>Est. true positives<br><small>(nudge count × 42.6% precision)</small></p></div>
      <div class="biz-num"><h3>{{ converted }}</h3><p>Est. conversions<br><small>(true positives × 20% conversion)</small></p></div>
      <div class="biz-num"><h3>{{ cost_avoidance }}</h3><p>Est. ED cost avoided<br><small>(conversions × AUD $800)</small></p></div>
    </div>
    <p class="biz-note">
      Assumptions: 20% of nudged true-positive members book an allied health visit; average ED presentation cost avoided = AUD $800 (conservative).
      Actual avoided costs include downstream inpatient admissions and chronic disease management savings not captured here.
      Precision figure (42.6%) from model evaluation on held-out test set.
    </p>
  </div>

  <h2>Model Drivers</h2>
  <div class="charts">
    <div class="chart wide">{{ charts.shap }}</div>
  </div>

  <h2>Model Performance</h2>
  <div class="model">
    <div class="metrics">
      <div class="metric"><h3>{{ roc_auc }}</h3><p>ROC-AUC</p></div>
      <div class="metric"><h3>{{ pr_auc }}</h3><p>PR-AUC</p></div>
      <div class="metric"><h3>{{ precision_top20 }}</h3><p>Precision @ Top 20%</p></div>
      <div class="metric"><h3>{{ recall_top20 }}</h3><p>Recall @ Top 20%</p></div>
    </div>
    <p style="font-size:13px;color:#636e72;margin-top:12px;">
      The model correctly ranks a high-risk member above a low-risk member 71% of the time.
      When the top 20% of members by score are flagged, 43% are true positives.
      Top predictors: comorbidity count, MSK flag, condition cluster, allied health utilisation in the past 6 months.
      <strong>Note:</strong> proof of concept on synthetic data. Val AUC ceiling ~0.727 reflects synthetic label construction —
      significantly higher AUC expected on real forward-looking claims data.
    </p>
  </div>
</div>

<footer>Allied Health Nudge &middot; Proof of concept &middot; Synthetic data only &middot; Not for clinical use</footer>
</body>
</html>
"""

def fmt_num(n):
    return f"{int(n):,}"

env = Environment()
template = env.from_string(HTML_TEMPLATE)

html = template.render(
    total_scored    = fmt_num(summary['total_scored']),
    high_risk       = fmt_num(summary['high_risk']),
    medium_risk     = fmt_num(summary['medium_risk']),
    nudge_count     = fmt_num(summary['nudge_signal']),
    nudge_rate      = f"{summary['nudge_rate']:.1%}",
    cost_avoidance  = f"${cost_avoidance:,}",
    true_positives  = fmt_num(true_positives),
    converted       = fmt_num(round(true_positives * conversion_rate)),
    roc_auc         = metrics['roc_auc'],
    pr_auc          = metrics['pr_auc'],
    precision_top20 = metrics['precision_top20pct'],
    recall_top20    = metrics['recall_top20pct'],
    causal_chain_svg= causal_chain_svg,
    charts          = charts,
)

report_path = outputs_dir / 'report.html'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f"Saved: {report_path}")
print(f"File size: {report_path.stat().st_size / 1024:.0f} KB")